# Notebook 19 — WU-19: Preprocessing correction protocol and read-only preflight

**CPU only · No training · No dataset changes.** This notebook verifies frozen 149-patient folds, 745 raw intensity audit rows, 745 normalized tensor audit rows and Dataset505 nnU-Net normalization metadata. It then produces a private **design-only** Dataset506/507 protocol and fold-specific lists of patients allowed for fitting.

**Important:** Training is explicitly *blocked* until fold-specific fitted parameters, image-only anatomical supports, transform application and matching nnU-Net inference all pass WU20. This notebook cannot accidentally start GPU training.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Validate inputs and download immutable audit source

No patient-level data are uploaded to GitHub. The notebook reads the two completed prior audit tables and an immutable splits JSON. Outputs go to a **new** folder in Drive. It does not write Dataset504/505.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, subprocess, sys
ROOT = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
SPLITS = ROOT / 'nnUNet_preprocessed/Dataset505_ISLES2024_NCCT_PERFUSION/splits_final.json'
RAW_AUDIT = ROOT / 'results/perfusion_intensity_audit/isles2024_perfusion_case_intensity_audit_PRIVATE.csv'
TENSOR_AUDIT = ROOT / 'results/perfusion_preprocessed_tensor_audit/isles2024_preprocessed_tensor_audit_PRIVATE.csv'
PLANS = ROOT / 'nnUNet_preprocessed/Dataset505_ISLES2024_NCCT_PERFUSION/nnUNetPlans.json'
OUTPUT = ROOT / 'results/preprocessing_protocol_wu19'
LOG = OUTPUT / 'wu19_preflight.log'
SCRIPT = Path('/content/plan_isles2024_wu19_ablation.py')
SCRIPT_COMMIT = '776f3dd1635e5b13b893c6d49a466ef8735e4bbf'
OUTPUT.mkdir(parents=True, exist_ok=True)
for item in (SPLITS, RAW_AUDIT, TENSOR_AUDIT, PLANS):
    if not item.is_file(): raise FileNotFoundError(item)

def run_logged(label, args):
    cmd=[str(x) for x in args]
    print('===',label,'===')
    with LOG.open('a',encoding='utf8') as f:
        f.write(f'\n[{datetime.now(timezone.utc).isoformat()}] {label}: $ {" ".join(cmd)}\n')
        with subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:
            for line in p.stdout:
                print(line,end='',flush=True);f.write(line);f.flush()
            rc=p.wait()
        f.write(f'[exit_code={rc}]\n')
    if rc:raise RuntimeError(f'{label} failed; read {LOG}')

run_logged('Download pinned WU19 preflight',[
    'curl','-fLsS','--retry','4','--retry-delay','3',
    f'https://raw.githubusercontent.com/NTinkicht/HybridStrokeSeg-Reproduction/{SCRIPT_COMMIT}/scripts/plan_isles2024_wu19_ablation.py',
    '-o', SCRIPT])
assert SCRIPT.is_file() and SCRIPT.stat().st_size > 12000
print('Pinned source commit:',SCRIPT_COMMIT)
print('Private output folder:',OUTPUT)


## 2. Run all WU-19 gates (fast CPU-only)

This verifies all existing audits, writes a private patient/fold manifest, aggregate JSON and a Markdown protocol. No new images are generated. It never changes an existing model or mask.


In [ ]:
run_logged('WU19 reproducible preprocessing protocol',[
    sys.executable,'-u',SCRIPT,'--splits',SPLITS,
    '--raw-audit-csv',RAW_AUDIT,'--preprocessed-audit-csv',TENSOR_AUDIT,
    '--plans505',PLANS,'--output-dir',OUTPUT])


## 3. Verify the design is ready for review, **not** for GPU training

The expected state is **DESIGN_ONLY_NOT_APPROVED_FOR_TRAINING**. WU20 will implement fold-specific training/inference and verify transform behavior before any corrected model starts.


In [ ]:
plan=json.loads((OUTPUT/'WU19_preprocessing_protocol_PRIVATE.json').read_text())
assert plan['training_ready'] is False
assert plan['status']=='DESIGN_ONLY_NOT_APPROVED_FOR_TRAINING'
assert len(plan['folds'])==5
assert sum(f['n_validation'] for f in plan['folds'])==149
assert len(plan['targets'])==2
assert plan['qcs_from_frozen_audits']['MTT']['pre_patients_abs_z_gt_100']==125
print('WU19 CHECKS PASS: 149 patients / 5 frozen folds / 745 raw audit rows / 745 tensor audit rows')
print('Dataset506/507: DESIGN ONLY. GPU training: BLOCKED pending WU20 gates.')
print('Protocol:', OUTPUT/'WU19_protocol_readme.md')
print('Private fold manifests:', OUTPUT/'WU19_preprocessing_protocol_PRIVATE.json')
